<a href="https://colab.research.google.com/github/mAliAytekin/ai-research-notes/blob/main/mlp_with_ga.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## import some important things

In [159]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from dataclasses import dataclass

## load data & train-test split

In [160]:
iris = load_iris()
X = iris.data
y = iris.target

In [161]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [162]:
X_train[:3]

array([[4.4, 2.9, 1.4, 0.2],
       [4.9, 2.5, 4.5, 1.7],
       [6.8, 2.8, 4.8, 1.4]])

In [163]:
y_train[:3]

array([0, 2, 1])

## standardization

In [164]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [165]:
X_train[:3]

array([[-1.72156775, -0.33210111, -1.34572231, -1.32327558],
       [-1.12449223, -1.22765467,  0.41450518,  0.6517626 ],
       [ 1.14439475, -0.5559895 ,  0.58484978,  0.25675496]])

In [166]:
y_train[:3]

array([0, 2, 1])

In [167]:
# convert to pytorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

In [168]:
X_train_tensor[:3]

tensor([[-1.7216, -0.3321, -1.3457, -1.3233],
        [-1.1245, -1.2277,  0.4145,  0.6518],
        [ 1.1444, -0.5560,  0.5848,  0.2568]])

In [169]:
y_train_tensor[:3]

tensor([0, 2, 1])

## multi layer perceptron

In [170]:
@dataclass
class MLPConfig:
    input_dim: int = 4
    hidden_dim: int = 10
    output_dim: int = 3
    learning_rate: float = 0.01
    num_epochs: int = 100

In [171]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(MLPConfig.input_dim, MLPConfig.hidden_dim)
        self.fc2 = nn.Linear(MLPConfig.hidden_dim, MLPConfig.output_dim)

    def forward(self,x):
      x = torch.relu(self.fc1(x))
      x = self.fc2(x)
      return x

    def set_weights(self, weights):
        # set weights with vectors from genetic algorithm
        # there is no gradient !!!
        with torch.no_grad():
          fc1_size = self.fc1.weight.numel() + self.fc1.bias.numel()
          fc1_weights = weights[:fc1_size]

          # fc1.weight : (hidden_size, input_size)
          self.fc1.weight.data = fc1_weights[:self.fc1.weight.numel()].reshape(self.fc1.weight.shape)

          # fc1.bias : (hidden_size,)
          self.fc1.bias.data = fc1_weights[self.fc1.weight.numel():].reshape(self.fc1.bias.shape)

          # settings for fc2
          fc2_size = self.fc2.weight.numel()
          fc2_weights = weights[fc1_size:]

          # fc2.weight: (output_size, hidden_size)
          self.fc2.weight.data = fc2_weights[:fc2_size].reshape(
              self.fc2.weight.shape
          )

           # fc2.bias: (output_size,)
          self.fc2.bias.data = fc2_weights[fc2_size:].reshape(
              self.fc2.bias.shape
          )

    def get_weights_as_vector(self):
      weights = []
      weights.append(self.fc1.weight.data.flatten())
      weights.append(self.fc1.bias.data.flatten())
      weights.append(self.fc2.weight.data.flatten())
      weights.append(self.fc2.bias.data.flatten())
      return torch.cat(weights)


## genetic algorithm

In [172]:
@dataclass
class GeneticAlgorithmConfig:
  population_size : int = 50
  mutation_rate : float = 0.1
  crossover_rate : float = 0.8
  num_generations : int = 100
  regularization_coeff : float = 0.001
  tournament_size : int = 3
  crossover_method : str = 'heuristic'
  elitism_count : int = 2
  uniform_rate : float = 0.5
  convergence_rate : float = 0.4

In [173]:
class GeneticAlgorithm:
  def __init__(self,model):
    self.model = model

    # total weights count of model
    self.chromosome_length  = sum(p.numel() for p in model.parameters())

    # init population
    self.population = self.initialization_population()

  def initialization_population(self):
    population = []
    for _ in range(GeneticAlgorithmConfig.population_size):
      # rand weight between -1 and 1
      chromosome = torch.rand(self.chromosome_length )*0.1
      population.append(chromosome)
    return population

  def fitness(self,chromosome):
    # set weights at model
    self.model.set_weights(chromosome)

    with torch.no_grad():
      outputs = self.model(X_train_tensor)
      predictions = torch.argmax(outputs, dim=1)
      accuracy = (predictions == y_train_tensor).float().mean()

    # fitness = accuracy - regularization coeff * euclidean norm
    weight_penalty = GeneticAlgorithmConfig.regularization_coeff * torch.norm(chromosome).item()
    return accuracy - weight_penalty

  def evaluate_population(self):
    fitness_scores = []
    for chromosome in self.population:
      fitness_scores.append(self.fitness(chromosome))
    return torch.tensor(fitness_scores)

  def selection(self,fitness_scores):
    selected = []

    for _ in range(GeneticAlgorithmConfig.population_size // 2):
      indices = torch.randint(0,len(self.population),(GeneticAlgorithmConfig.tournament_size,))
      tournament_fitness = fitness_scores[indices]
      winner_index = indices[torch.argmax(tournament_fitness)]
      selected.append(self.population[winner_index].clone())

    return selected

  def _uniform_crossover(self,parent1,parent2):
    mask = torch.rand(len(parent1)) < GeneticAlgorithmConfig.uniform_rate

    child1 = torch.where(mask,parent1,parent2)
    child2 = torch.where(mask,parent2,parent1)

    return child1,child2

  def _single_point_crossover(self,parent1,parent2):
    crossover_point = torch.randint(1, self.chromosome_length - 1, (1,)).item()

    child1 = torch.cat([parent1[:crossover_point], parent2[crossover_point:]])
    child2 = torch.cat([parent2[:crossover_point], parent1[crossover_point:]])

    return child1, child2

  def _heuristic_crossover(self,parent1,parent2):
    fitness1 = self.fitness(parent1)
    fitness2 = self.fitness(parent2)

    if fitness1 > fitness2:
      better,worse = parent1,parent2
    else:
      better,worse = parent2,parent1

    child1 = better + GeneticAlgorithmConfig.convergence_rate*(better-worse)
    child2 = worse + GeneticAlgorithmConfig.convergence_rate*(better-worse)

    return child1,child2

  def crossover(self,parent1,parent2):
    if torch.rand(1).item() > GeneticAlgorithmConfig.crossover_rate:
      return parent1.clone(),parent2.clone()

    if GeneticAlgorithmConfig.crossover_method == 'uniform':
      return self._uniform_crossover(parent1,parent2)
    elif GeneticAlgorithmConfig.crossover_method == 'single_point':
      return self._single_point_crossover(parent1,parent2)
    else :
      return self._heuristic_crossover(parent1,parent2)

  def mutation(self,chromosome):
    mutated = chromosome.clone()

    for i in range(len(mutated)):
      if torch.rand(1).item() < GeneticAlgorithmConfig.mutation_rate:
        mutated[i] += torch.randn(1).item() * 0.05

    return mutated

  def evolve(self):
      best_fitness_history = []
      avg_fitness_history = []

      for generation in range(GeneticAlgorithmConfig.num_generations):
          fitness_scores = self.evaluate_population()

          best_fitness = fitness_scores.max().item()
          avg_fitness = fitness_scores.mean().item()
          best_fitness_history.append(best_fitness)
          avg_fitness_history.append(avg_fitness)

          if generation % 10 == 0:
              print(f"Generation {generation}: Best = {best_fitness:.4f}, Avg = {avg_fitness:.4f}")

          selected = self.selection(fitness_scores)

          new_population = []

          elite_indices = torch.argsort(fitness_scores, descending=True)[:GeneticAlgorithmConfig.elitism_count]
          for idx in elite_indices:
              new_population.append(self.population[idx].clone())

          while len(new_population) < GeneticAlgorithmConfig.population_size:

              idx1 = torch.randint(0, len(selected), (1,)).item()
              idx2 = torch.randint(0, len(selected), (1,)).item()

              parent1 = selected[idx1]
              parent2 = selected[idx2]

              child1, child2 = self.crossover(parent1, parent2)

              child1 = self.mutation(child1)
              child2 = self.mutation(child2)

              new_population.extend([child1, child2])

          self.population = new_population[:GeneticAlgorithmConfig.population_size]

      final_fitness = self.evaluate_population()
      best_idx = torch.argmax(final_fitness)
      best_chromosome = self.population[best_idx]

      return best_chromosome, best_fitness_history, avg_fitness_history

In [174]:
mlp = MLP()
ga = GeneticAlgorithm(mlp)
best_chromosome, best_fitness_history, avg_fitness_history = ga.evolve()

Generation 0: Best = 0.6578, Avg = 0.3438
Generation 10: Best = 0.7993, Avg = 0.6223
Generation 20: Best = 0.8992, Avg = 0.7145
Generation 30: Best = 0.9824, Avg = 0.8597
Generation 40: Best = 0.9824, Avg = 0.8545
Generation 50: Best = 0.9907, Avg = 0.9051
Generation 60: Best = 0.9907, Avg = 0.9225
Generation 70: Best = 0.9907, Avg = 0.8866
Generation 80: Best = 0.9907, Avg = 0.9098
Generation 90: Best = 0.9907, Avg = 0.8667


In [175]:
mlp.set_weights(best_chromosome)

In [176]:
with torch.no_grad():
    outputs = mlp(X_test_tensor)
    predictions = torch.argmax(outputs, dim=1)
    accuracy = (predictions == y_test_tensor).float().mean().item()
print(f"\nGenetic Algorithm Test Accuracy: {accuracy * 100:.2f}%")


Genetic Algorithm Test Accuracy: 90.00%


In [177]:
print("Comparing with Backpropagation ...")
backprop_model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(backprop_model.parameters(), lr=MLPConfig.learning_rate)

backprop_losses = []
for epoch in range(MLPConfig.num_epochs):
    optimizer.zero_grad()
    outputs = backprop_model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

    backprop_losses.append(loss.item())

    if epoch % 20 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")


with torch.no_grad():
    outputs = backprop_model(X_test_tensor)
    predictions = torch.argmax(outputs, dim=1)
    backprop_accuracy = (predictions == y_test_tensor).float().mean().item()

print(f"\nBackpropagation Test Accuracy: {backprop_accuracy * 100:.2f}%")

Comparing with Backpropagation ...
Epoch 0: Loss = 1.0322
Epoch 20: Loss = 0.4587
Epoch 40: Loss = 0.2360
Epoch 60: Loss = 0.1418
Epoch 80: Loss = 0.0972

Backpropagation Test Accuracy: 93.33%
